# Stage 2 v2 — Augment Dataset via items_tv_v8 (intermediate)

**Khac biet so voi 08_augment_dataset_v4.ipynb (v1):**
- v1: tv_v7 → LLM → build prompts truc tiep → items_prompts_tv_4
- **v2 (notebook nay):** tv_v7 → LLM → **items_tv_v8** (luu summary_version2) → items_prompts_tv_5

**items_tv_v8 schema:** tat ca cot cua v7 + cot `summary_version2`
- `summary_version2` = `header goc` (Tieu de/Danh muc/Thuong hieu giu nguyen) + `body moi` (Mo ta/Thong so do LLM viet lai)
- Train split: ~185K rows (expand theo price-bucket multiplier 5x/3x/2x/1x/4x)
- Val/Test split: giu nguyen tu v7 (khong augment)

**items_prompts_tv_5:** format giong het tv_4 (prompt, completion, price_vnd_true)
- Train: 85K (tu tv_3 goc) + 185K (tu items_tv_v8 summary_version2) = ~270K

In [6]:
import os
import re
import sys
import json
import time
import pickle
import random
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from dataclasses import dataclass
from typing import Optional

from datasets import load_dataset, DatasetDict, Dataset
from dotenv import load_dotenv
from huggingface_hub import login
from groq import Groq

NOTEBOOK_DIR = Path(".")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- HF datasets ---
SOURCE_TV7    = "SeanSunny/items_tv_v7"
OUTPUT_TV8    = "SeanSunny/items_tv_v8"
SOURCE_TV3    = "SeanSunny/items_prompts_tv_3"
OUTPUT_TV5    = "SeanSunny/items_prompts_tv_5"

MAX_PRICE    = 1_000_000
QUESTION_FULL = "S\u1ea3n ph\u1ea9m n\u00e0y c\u00f3 gi\u00e1 bao nhi\u00eau ?"
PRICE_PREF   = "\n\nGi\u00e1 l\u00e0: "

# --- Folders (khac voi v1 de tranh xung dot) ---
BATCHES_FOLDER = NOTEBOOK_DIR / "batches_aug_v8"
OUTPUT_FOLDER  = NOTEBOOK_DIR / "output_aug_v8"
STATE_FILE     = NOTEBOOK_DIR / "batches_aug_v8.pkl"
BATCHES_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# --- env ---
env_path = NOTEBOOK_DIR.parent / ".env"
load_dotenv(env_path)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
GROQ_KEY = os.environ.get("GROQ_API_KEY", "")
if HF_TOKEN:
    login(HF_TOKEN)
    print("HF login OK")
else:
    raise RuntimeError("HF_TOKEN not set")
if not GROQ_KEY:
    raise RuntimeError("GROQ_API_KEY not set")
groq_client = Groq(api_key=GROQ_KEY)
print("Groq client OK | Imports OK")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK
Groq client OK | Imports OK


## 1. Load items_tv_v7

In [7]:
print(f"Loading {SOURCE_TV7}...")
ds_v7 = load_dataset(SOURCE_TV7)
train_v7 = list(ds_v7["train"])
val_v7   = list(ds_v7["validation"])
test_v7  = list(ds_v7["test"])
print(f"Train: {len(train_v7):,} | Val: {len(val_v7):,} | Test: {len(test_v7):,}")

# Sanity check
s = train_v7[0]
print(f"Columns: {list(s.keys())}")
assert s["full"],    "full is None — check items_tv_v7"
assert s["summary"], "summary is None — check items_tv_v7"
print(f"Sample title  : {s['title'][:80]}")
print(f"Sample summary: {s['summary'][:120]}")


Loading SeanSunny/items_tv_v7...


DatasetNotFoundError: Dataset 'SeanSunny/items_tv_v7' doesn't exist on the Hub or cannot be accessed.

## 2. Filter + parse_summary

In [ ]:
train_filtered = [row for row in train_v7 if row["price"] <= MAX_PRICE]
print(f"Train after price filter: {len(train_filtered):,} / {len(train_v7):,}")

def parse_summary(summary: str):
    """Returns (header, body) or None. header=3 lines, body=2 lines."""
    if not summary:
        return None
    header_lines, body_lines = [], []
    for line in summary.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        if line.startswith("Ti\u00eau \u0111\u1ec1:") or line.startswith("Tieu de:"):
            header_lines.append(line)
        elif line.startswith("Danh m\u1ee5c:") or line.startswith("Danh muc:"):
            header_lines.append(line)
        elif line.startswith("Th\u01b0\u01a1ng hi\u1ec7u:") or line.startswith("Thuong hieu:"):
            header_lines.append(line)
        elif line.startswith("M\u00f4 t\u1ea3:") or line.startswith("Mo ta:"):
            body_lines.append(line)
        elif line.startswith("Th\u00f4ng s\u1ed1:") or line.startswith("Thong so:"):
            body_lines.append(line)
    if len(header_lines) == 3 and len(body_lines) == 2:
        return "\n".join(header_lines), "\n".join(body_lines)
    return None

ok = fail = 0
for row in train_filtered[:2000]:
    (ok := ok+1) if parse_summary(row["summary"]) else (fail := fail+1)
print(f"Parse check (2000): OK={ok} FAIL={fail}")
if fail == 0:
    h, b = parse_summary(train_filtered[0]["summary"])
    print(f"\nSample header:\n{h}")
    print(f"\nSample body:\n{b}")


## 3. Price bucket + multiplier A5

In [ ]:
BUCKETS = [
    ("<50K",     0,        50_000,  5),
    ("50-100K",  50_000,  100_000,  3),
    ("100-200K", 100_000, 200_000,  2),
    ("200-500K", 200_000, 500_000,  1),
    ("500K-1M",  500_000, 1_000_001, 4),
]
prices = np.array([row["price"] for row in train_filtered])
bucket_multipliers = {}
total_aug = 0
print(f"{'Bucket':<12} {'Items':>7} {'%':>6} {'Mult':>5} {'Aug':>9}")
print("-" * 45)
for name, lo, hi, mult in BUCKETS:
    mask = (prices >= lo) & (prices < hi)
    count = int(mask.sum())
    aug = count * mult
    total_aug += aug
    print(f"{name:<12} {count:>7,} {count/len(prices)*100:>5.1f}%  {mult:>3}x  {aug:>8,}")
    for idx in np.where(mask)[0]:
        bucket_multipliers[int(idx)] = mult
print("-" * 45)
print(f"Total augmented rows : {total_aug:,}")
print(f"Total items_tv_v8 train: {total_aug:,} (augmented only — orig not repeated here)")
print(f"Total items_prompts_tv5 train: {len(train_filtered) + total_aug:,}")
assert len(bucket_multipliers) == len(train_filtered)


## 4. AugBatchManager + SYSTEM_PROMPT

Giong het v1 — khac folder names (`batches_aug_v8/`, `output_aug_v8/`).

In [ ]:
SYSTEM_PROMPT_AUG = """Dựa vào thông tin sản phẩm gốc và tóm tắt hiện tại, hãy viết lại 'Mô tả' và 'Thông số' theo cách khác.
Yêu cầu:
- Giữ nguyên ý nghĩa gốc, nhưng sử dụng từ ngữ và cách diễn đạt mới mẻ, tự nhiên.
- Tuyệt đối KHÔNG thay đổi các thông số định lượng.
- Không được viết trùng lặp với nội dung 'Mô tả' hoặc 'Thông số' hiện tại.
- Chỉ trả về đúng 2 dòng theo định dạng sau, không thêm bất kỳ lời dẫn hay ký tự thừa nào:
Mô tả: [1 câu mô tả ngắn gọn về sản phẩm]
Thông số: [1 câu về tính năng hoặc đặc điểm kỹ thuật nổi bật]"""


In [ ]:
MODEL = "openai/gpt-oss-20b"
BATCH_SIZE = 1_000

def build_user_message(full_text: str, body: str) -> str:
    return f"Thong tin san pham goc:\n{full_text}\n\n---\nTom tat hien tai:\n{body}"

def parse_aug_output(llm_text: str):
    """Returns (mo_ta_text, thong_so_text) or None."""
    mo_ta = thong_so = None
    for line in llm_text.strip().split("\n"):
        line = line.strip()
        if line.startswith("M\u00f4 t\u1ea3:") or line.startswith("Mo ta:"):
            mo_ta = re.sub(r'^(M[o\u00f4] t[a\u1ea3]):?\s*', '', line).strip()
        elif line.startswith("Th\u00f4ng s\u1ed1:") or line.startswith("Thong so:"):
            thong_so = re.sub(r'^(Th[o\u00f4]ng s[o\u1ed1]):?\s*', '', line).strip()
    if mo_ta and thong_so:
        return mo_ta, thong_so
    return None


@dataclass
class AugBatch:
    start: int
    end: int
    filename: str
    file_id: Optional[str] = None
    batch_id: Optional[str] = None
    output_file_id: Optional[str] = None
    done: bool = False


class AugBatchManager:
    batches: list = []
    request_list: list = []

    @classmethod
    def build_requests(cls, filtered_items, multipliers):
        cls.request_list = []
        skipped = 0
        for idx, row in enumerate(tqdm(filtered_items, desc="Building requests")):
            result = parse_summary(row["summary"])
            if result is None:
                skipped += 1
                continue
            _, body = result
            full_text = row["full"] or ""
            for v in range(multipliers[idx]):
                cls.request_list.append((idx, v, build_user_message(full_text, body)))
        print(f"Requests: {len(cls.request_list):,} | Skipped: {skipped}")

    @classmethod
    def create_batches(cls):
        cls.batches = []
        for start in range(0, len(cls.request_list), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(cls.request_list))
            cls.batches.append(AugBatch(start, end, f"aug_{start}_{end}.jsonl"))
        print(f"Created {len(cls.batches)} batches")

    @classmethod
    def _make_jsonl_line(cls, item_idx, version, user_msg):
        return json.dumps({
            "custom_id": f"{item_idx}_{version}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": MODEL,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT_AUG},
                    {"role": "user",   "content": user_msg},
                ],
                "reasoning_effort": "low",
            },
        }, ensure_ascii=False)

    @classmethod
    def write_and_submit(cls, batch):
        fpath = BATCHES_FOLDER / batch.filename
        with fpath.open("w", encoding="utf-8") as f:
            for item_idx, v, user_msg in cls.request_list[batch.start:batch.end]:
                f.write(cls._make_jsonl_line(item_idx, v, user_msg) + "\n")
        with fpath.open("rb") as f:
            resp = groq_client.files.create(file=f, purpose="batch")
        batch.file_id = resp.id
        resp2 = groq_client.batches.create(
            completion_window="24h",
            endpoint="/v1/chat/completions",
            input_file_id=batch.file_id,
        )
        batch.batch_id = resp2.id

    @classmethod
    def run(cls):
        for batch in tqdm(cls.batches, desc="Submitting"):
            if not batch.batch_id:
                cls.write_and_submit(batch)
        print(f"Submitted {len(cls.batches)} batches")

    @classmethod
    def fetch(cls):
        for batch in cls.batches:
            if batch.done:
                continue
            result = groq_client.batches.retrieve(batch.batch_id)
            if result.status == "completed":
                batch.output_file_id = result.output_file_id
                groq_client.files.content(result.output_file_id).write_to_file(
                    str(OUTPUT_FOLDER / batch.filename)
                )
                batch.done = True
        finished = sum(1 for b in cls.batches if b.done)
        print(f"Finished {finished} / {len(cls.batches)} batches")
        return finished

    @classmethod
    def save(cls):
        with STATE_FILE.open("wb") as f:
            pickle.dump(cls.batches, f)
        print(f"State saved: {len(cls.batches)} batches → {STATE_FILE}")

    @classmethod
    def load(cls):
        with STATE_FILE.open("rb") as f:
            cls.batches = pickle.load(f)
        print(f"State loaded: {len(cls.batches)} batches")

    @classmethod
    def resubmit_failed(cls):
        resubmitted = 0
        for batch in cls.batches:
            if batch.done:
                continue
            result = groq_client.batches.retrieve(batch.batch_id)
            if result.status in ("failed", "expired", "cancelled"):
                cls.write_and_submit(batch)
                resubmitted += 1
                time.sleep(0.5)
        if resubmitted:
            cls.save()
        print(f"Resubmitted {resubmitted} batches")
        return resubmitted


print("AugBatchManager OK | MODEL:", MODEL)


NameError: name 'dataclass' is not defined

## 5. Test don le — 3 items tu cac bucket khac nhau

In [ ]:
for tidx in [0, 1000, 50000]:
    row = train_filtered[tidx]
    result = parse_summary(row["summary"])
    if result is None:
        print(f"[{tidx}] SKIP: bad summary")
        continue
    header, body = result
    resp = groq_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_AUG},
            {"role": "user",   "content": build_user_message(row["full"] or "", body)},
        ],
        reasoning_effort="low",
    )
    llm_out = resp.choices[0].message.content
    parsed = parse_aug_output(llm_out)
    if parsed:
        mo_ta, thong_so = parsed
        new_body = f"M\u00f4 t\u1ea3: {mo_ta}\nTh\u00f4ng s\u1ed1: {thong_so}"
        summary_v2 = f"{header}\n{new_body}"
        print(f"\n[{tidx}] {row['title'][:60]}")
        print(f"  ORIG body : {body}")
        print(f"  NEW  body : {new_body}")
        print(f"  summary_v2:\n{summary_v2}")
        print(f"  Tokens: {resp.usage.prompt_tokens}in/{resp.usage.completion_tokens}out")
    else:
        print(f"[{tidx}] PARSE FAIL: {repr(llm_out[:80])}")


## 6. Test batch — 15 samples

**[USER]** Kiem tra chat luong. Neu OK → chay Full batch.
Neu can chinh SYSTEM_PROMPT → sua cell tren roi chay lai.

In [ ]:
TEST_N = 15
TEST_IDXS = list(range(0, len(train_filtered), len(train_filtered) // TEST_N))[:TEST_N]

test_jsonl = BATCHES_FOLDER / "test_batch.jsonl"
test_reqs = []
for idx in TEST_IDXS:
    row = train_filtered[idx]
    result = parse_summary(row["summary"])
    if result is None:
        continue
    _, body = result
    test_reqs.append((idx, 0, build_user_message(row["full"] or "", body)))

with test_jsonl.open("w", encoding="utf-8") as f:
    for item_idx, v, user_msg in test_reqs:
        f.write(AugBatchManager._make_jsonl_line(item_idx, v, user_msg) + "\n")

with test_jsonl.open("rb") as f:
    tf = groq_client.files.create(file=f, purpose="batch")
tb = groq_client.batches.create(
    completion_window="24h",
    endpoint="/v1/chat/completions",
    input_file_id=tf.id,
)
print(f"Test batch submitted: {tb.id} | Requests: {len(test_reqs)}")


In [ ]:
test_out_path = OUTPUT_FOLDER / "test_batch.jsonl"
while True:
    r = groq_client.batches.retrieve(tb.id)
    if r.status == "completed":
        groq_client.files.content(r.output_file_id).write_to_file(str(test_out_path))
        print("Test batch DONE")
        break
    elif r.status in ("failed", "expired", "cancelled"):
        raise RuntimeError(f"Test batch {r.status}")
    print(f"Status: {r.status} — waiting 15s...")
    time.sleep(15)


In [ ]:
# Hien thi ket qua: original body vs summary_version2
parse_fail = 0
print(f"=== Test Batch Results ({TEST_N} samples) ===\n")

with test_out_path.open(encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        item_idx = int(obj["custom_id"].split("_")[0])
        llm_text = obj["response"]["body"]["choices"][0]["message"]["content"]
        row = train_filtered[item_idx]
        header, orig_body = parse_summary(row["summary"])
        parsed = parse_aug_output(llm_text)
        if parsed is None:
            parse_fail += 1
            print(f"[{item_idx}] PARSE FAIL: {repr(llm_text[:80])}")
            continue
        mo_ta, thong_so = parsed
        new_body = f"M\u00f4 t\u1ea3: {mo_ta}\nTh\u00f4ng s\u1ed1: {thong_so}"
        summary_v2 = f"{header}\n{new_body}"
        print(f"[{item_idx}] {row['title'][:55]} | {row['price']:,} VND")
        print(f"  ORIG: {orig_body.replace(chr(10), ' | ')}")
        print(f"  V2  : {new_body.replace(chr(10), ' | ')}")
        print()

print(f"Parse failures: {parse_fail}/{TEST_N}")
print("\n[USER] Kiem tra summary_version2 o tren.")
print("       Neu tot → chay Full batch.")
print("       Neu can chinh → sua SYSTEM_PROMPT_AUG roi chay lai.")


## 7. Full batch — ~185K requests

**[USER] Chi chay sau khi confirm test batch OK.**

In [ ]:
AugBatchManager.build_requests(train_filtered, bucket_multipliers)
AugBatchManager.create_batches()


In [ ]:
AugBatchManager.run()


In [ ]:
# QUAN TRONG: save ngay sau submit
AugBatchManager.save()


In [ ]:
while True:
    finished = AugBatchManager.fetch()
    if finished == len(AugBatchManager.batches):
        print("All batches DONE!")
        break
    print(f"Waiting 60s... ({finished}/{len(AugBatchManager.batches)})")
    time.sleep(60)


In [ ]:
AugBatchManager.save()


### Resume (neu kernel crash)
```python
# AugBatchManager.load()
# AugBatchManager.build_requests(train_filtered, bucket_multipliers)
# AugBatchManager.fetch()
```

In [ ]:
# Resubmit neu co batch failed/expired
AugBatchManager.resubmit_failed()


## 8. Parse + build items_tv_v8

Moi (item_idx, version) → reconstruct `summary_version2 = header + new_body`.
`items_tv_v8` train = expanded rows: moi row la 1 augmented version cua 1 item goc.
Cot `summary` = original (tu v7). Cot `summary_version2` = LLM rewritten.

In [ ]:
# Doc tat ca LLM outputs
aug_results = {}  # (item_idx, version) -> llm_text
for batch in tqdm(AugBatchManager.batches, desc="Reading outputs"):
    out_path = OUTPUT_FOLDER / batch.filename
    if not out_path.exists():
        print(f"WARNING: missing {batch.filename}")
        continue
    with out_path.open(encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            cid = obj["custom_id"]
            item_idx, version = int(cid.split("_")[0]), int(cid.split("_")[1])
            aug_results[(item_idx, version)] = obj["response"]["body"]["choices"][0]["message"]["content"]

print(f"LLM outputs: {len(aug_results):,} / {len(AugBatchManager.request_list):,}")
print(f"Missing: {len(AugBatchManager.request_list) - len(aug_results)}")


In [ ]:
# Build items_tv_v8 rows (expanded)
# Schema: tat ca cot cua v7 + summary_version2
v8_rows = []
build_ok = build_fail = 0

for (item_idx, version), llm_text in tqdm(aug_results.items(), desc="Building v8 rows"):
    row = train_filtered[item_idx]
    result = parse_summary(row["summary"])
    if result is None:
        build_fail += 1
        continue
    header, _ = result
    parsed = parse_aug_output(llm_text)
    if parsed is None:
        build_fail += 1
        continue
    mo_ta, thong_so = parsed
    new_body       = f"M\u00f4 t\u1ea3: {mo_ta}\nTh\u00f4ng s\u1ed1: {thong_so}"
    summary_v2     = f"{header}\n{new_body}"

    v8_rows.append({
        "title":            row["title"],
        "category":         row["category"],
        "price":            row["price"],
        "full":             row["full"],
        "brand":            row.get("brand"),
        "summary":          row["summary"],       # original tu v7
        "summary_version2": summary_v2,            # LLM rewritten
        "aug_version":      version,               # 0,1,2,... (de biet version thu may)
        "prompt":           None,
        "id":               None,
    })
    build_ok += 1

print(f"v8 rows built: {build_ok:,} OK | {build_fail} failed")
print(f"Sample summary_version2:\n{v8_rows[0]['summary_version2']}")


## 9. Push items_tv_v8

In [ ]:
# Val/Test: giu nguyen tu v7 (khong augment, them cot summary_version2=None)
def add_v2_col(rows):
    return [{**row, "summary_version2": None, "aug_version": None} for row in rows]

ds_v8 = DatasetDict({
    "train":      Dataset.from_list(v8_rows),
    "validation": Dataset.from_list(add_v2_col(val_v7)),
    "test":       Dataset.from_list(add_v2_col(test_v7)),
})

print(f"items_tv_v8:")
print(ds_v8)
print(f"Train columns: {ds_v8['train'].column_names}")
print(f"Pushing {OUTPUT_TV8}...")
ds_v8.push_to_hub(OUTPUT_TV8, private=True)
print(f"Pushed: https://huggingface.co/datasets/{OUTPUT_TV8}")


## 10. Build items_prompts_tv_5

`items_prompts_tv_5` = format giong het `items_prompts_tv_4`:
- Train: 85K goc (tu tv_3, dung `summary` goc) + 185K aug (tu items_tv_v8, dung `summary_version2`)
- Val / Test: giu nguyen tu tv_3

In [ ]:
# Load items_prompts_tv_3 (original prompts)
print(f"Loading {SOURCE_TV3}...")
ds_tv3 = load_dataset(SOURCE_TV3)
orig_train = list(ds_tv3["train"])
orig_val   = list(ds_tv3["val"])
orig_test  = list(ds_tv3["test"])
print(f"orig train={len(orig_train):,} | val={len(orig_val):,} | test={len(orig_test):,}")
assert set(orig_train[0].keys()) == {"prompt", "completion", "price_vnd_true"}
print("Schema OK")


In [ ]:
# Build augmented prompts tu items_tv_v8 summary_version2
aug_examples = []
for row in tqdm(v8_rows, desc="Building prompts tv5"):
    sv2 = row["summary_version2"]
    if not sv2:
        continue
    prompt = f"{QUESTION_FULL}\n{sv2}{PRICE_PREF}"
    aug_examples.append({
        "prompt":         prompt,
        "completion":     str(int(round(row["price"] / 1000))),
        "price_vnd_true": int(row["price"]),
    })

print(f"Augmented prompts built: {len(aug_examples):,}")

# Combine + shuffle
combined_train = orig_train + aug_examples
random.seed(SEED)
random.shuffle(combined_train)

print(f"Combined train: {len(orig_train):,} (orig) + {len(aug_examples):,} (aug) = {len(combined_train):,}")

# Quality check
empty_p = sum(1 for ex in combined_train if not ex["prompt"])
empty_c = sum(1 for ex in combined_train if not ex["completion"])
assert empty_p == 0 and empty_c == 0, f"Empty: {empty_p} prompts, {empty_c} completions"
prices_s = [ex["price_vnd_true"] for ex in combined_train]
print(f"Price range: {min(prices_s):,} — {max(prices_s):,} VND")
print("Quality check OK")


In [ ]:
ds_tv5 = DatasetDict({
    "train": Dataset.from_list(combined_train),
    "val":   Dataset.from_list(orig_val),
    "test":  Dataset.from_list(orig_test),
})

print(f"items_prompts_tv_5:")
print(ds_tv5)
print(f"Pushing {OUTPUT_TV5}...")
ds_tv5.push_to_hub(OUTPUT_TV5, private=True)
print(f"Pushed: https://huggingface.co/datasets/{OUTPUT_TV5}")
print(f"\nDone! Train size: {len(combined_train):,}")


## Summary

| Dataset | Split | Rows | Note |
|---|---|---|---|
| `items_tv_v8` | train | ~185K | augmented rows, cot `summary_version2` |
| `items_tv_v8` | val/test | 5K/5K | giu v7, `summary_version2=None` |
| `items_prompts_tv_5` | train | ~270K | 85K orig + 185K aug |
| `items_prompts_tv_5` | val/test | 3,926/3,872 | giu tv_3 |

**Buoc tiep:** Dung `items_prompts_tv_5` thay cho `tv_4` trong `06_train_v4_scratch.ipynb`
(doi `DATASET_NAME = "SeanSunny/items_prompts_tv_5"`).